# Example 1: Basic Globe (Topography only)

This notebook walks through the basic workflow of creating an uncolored 3D globe with surface topography. We will generate a sphere using a Fibonacci lattice, load global topography data from a NetCDF file, scale the displacement appropriately for 3D printing, and export the final watertight mesh as a binary STL file.

## Step 1: Import necessary libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from globe3d import (
    generate_sphere_points_fibonacci,
    load_netcdf_grid,
    calculate_displacement_scale,
    displace_vertices,
    write_stl_binary
)

## Step 2: Generate a base Fibonacci sphere

We define the physical size of our globe in millimeters. For this tutorial, we will use a radius of 40 mm (80 mm diameter). We will generate 5,000 vertices so that the notebook runs quickly (for high-quality production models, we recommend 50,000 to 200,000 vertices).

In [ ]:
model_radius_mm = 40.0
n_points = 5000

# Generate sphere vertices and faces
vertices, faces = generate_sphere_points_fibonacci(n_points=n_points, radius=model_radius_mm)
print(f"Generated sphere with {vertices.shape[0]} vertices and {faces.shape[0]} faces.")

## Step 3: Load the topography dataset (ETOPO)

We load global elevation data from the ETOPO NetCDF file. To keep interpolation fast in this notebook, we downsample the grid by selecting every 10th latitude and longitude cell.

In [ ]:
netcdf_path = "../inputs/ETOPO_2022_v1_60s_N90W180_surface.nc"

# Load latitude, longitude, and data grid arrays
lats, lons, grid = load_netcdf_grid(netcdf_path, lat_var='lat', lon_var='lon', data_var='z')

# Downsample grid
lats_ds = lats[::10]
lons_ds = lons[::10]
grid_ds = grid[::10, ::10]
print(f"Downsampled grid shape: {grid_ds.shape}")

## Step 4: Calculate displacement scale factor

Physical elevations are measured in meters, but the 3D printer uses millimeters. We scale elevations to fit our 40 mm radius model and exaggerate the heights by 40 times to make the topography feel tactile.

In [ ]:
vertical_exaggeration = 40.0
earth_radius_km = 6371.0

scale = calculate_displacement_scale(
    model_radius_mm=model_radius_mm,
    earth_radius_km=earth_radius_km,
    vertical_exagg=vertical_exaggeration
)
print(f"Scale factor: {scale}")

## Step 5: Apply radial displacement

Using the grid and scale, we translate each vertex of the sphere inward or outward along its normal vector.

In [ ]:
displaced_vertices = displace_vertices(
    vertices=vertices,
    lats=lats_ds,
    lons=lons_ds,
    grid=grid_ds,
    scale=scale,
    show_progress=True
)

## Step 6: Preview the result in 3D

We draw a scatter plot of a subset of the vertices to visualize the topography.

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot 1000 random points for speed
indices = np.random.choice(len(displaced_vertices), 1000, replace=False)
pts = displaced_vertices[indices]
sc = ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=pts[:, 2], cmap='terrain', s=4)
fig.colorbar(sc, ax=ax, label='Z coordinate (mm)')
ax.set_title("Basic Globe Topography preview")
plt.show()

## Step 7: Export to binary STL

We write the displaced model out to `../outputs/example_1_basic_globe.stl`, ready for slicing.

In [ ]:
output_dir = "../outputs"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "example_1_basic_globe.stl")

write_stl_binary(output_path, displaced_vertices, faces)
print(f"Saved STL file to: {output_path}")